# ThermoFuzzy-PID · 03 · Robustness & uncertainty

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fobos123deimos/thermofuzzy-pid/blob/main/notebooks/03_uncertainty_and_robustness.ipynb)

Monte Carlo stress-testing of gain-scheduled PID under uncertain thermal parameters, sensor noise and door disturbances; includes uncertainty-aware comparative metrics.

> **Data provenance:** experiments use deterministic synthetic thermal signals and parameters reported in a 2021 undergraduate thesis. No original sensor time series have been recovered; numerical results do not reproduce or validate the published curves.

## Read the model and method visually

[Six-page vector atlas](https://fobos123deimos.github.io/thermofuzzy-pid/assets/thermal-control-atlas.pdf) · [Gallery and editable TikZ](https://github.com/fobos123deimos/thermofuzzy-pid/blob/main/docs/DIAGRAMS.md)

These original explanatory diagrams distinguish source-reported parameters, reconstruction assumptions and synthetic evidence. Cabinet geometry and sensor positions are illustrative.

![Robustness: paired uncertainty and a separate linear loop audit](https://raw.githubusercontent.com/fobos123deimos/thermofuzzy-pid/main/docs/figures/tikz/05-robustness.png)

Matched synthetic uncertainty experiments and a separate fixed, linear, unsaturated sensitivity/margin analysis. The two evidence domains are not interchangeable.


In [ ]:
# Import local editable source, or install the repository in Google Colab.
from pathlib import Path
import sys, subprocess
candidates = [Path.cwd() / "src", Path.cwd().parent / "src"]
local_src = next((p for p in candidates if (p / "burnin_pid_fuzzy").exists()), None)
if local_src is not None:
    sys.path.insert(0, str(local_src.resolve()))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/fobos123deimos/thermofuzzy-pid.git"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from burnin_pid_fuzzy.models import (Plant, Scenario, DoorEvent, PLANTS_FROM_THESIS,
                                      fopdt_step_response, simulate, simulate_methods)
from burnin_pid_fuzzy.controllers import FuzzyPID, FixedPID, imc_pid_gains
from burnin_pid_fuzzy.fuzzy import FuzzyRuleBase, infer_normalized_gains, membership_degrees
from burnin_pid_fuzzy.metrics import evaluate
np.set_printoptions(precision=4, suppress=True)


## 1. Why uncertainty matters

The 2021 thesis points to uncertainty and robust disturbance rejection, but does not give raw parameter distributions. The ranges below are **illustrative hypotheses**, not confidence intervals or uncertainty estimates from the original identification campaign.

We perturb $K$, $\tau$ and $\theta$ by multiplicative factors and keep the IMC gains fixed at the nominal identified plant. We apply **identical plant draws and random seeds** to every controller for a fair paired comparison.

In [ ]:
rng=np.random.default_rng(2026)
nominal=PLANTS_FROM_THESIS["mean_temperature"]
n_trials=12 # raise to 50+ for deeper study; seeded for reproducibility
records=[]
for trial in range(n_trials):
    p=Plant(f"MC_{trial}",nominal.K*rng.uniform(.8,1.2),
            nominal.tau*rng.uniform(.7,1.3),nominal.delay*rng.uniform(.5,1.5),"assumed distribution")
    severity=-rng.uniform(3.5,6.)
    sc=Scenario(duration=2200,dt=2.,measurement_noise_std=.15,
                doors=(DoorEvent(800,60,severity),DoorEvent(1500,180,severity)))
    methods={
      "IMC-fixed-nominal":FixedPID(imc_pid_gains(),dt=sc.dt),
      "Thesis-sparse-fuzzy":FuzzyPID(mode="sparse",dt=sc.dt),
      "Dense-smoothed-fuzzy":FuzzyPID(mode="dense",smoothing=.18,dt=sc.dt)
    }
    for name,ctrl in methods.items():
        output=simulate(ctrl,p,sc,seed=trial)
        metrics=evaluate(output,sc)
        records.append({"trial":trial,"method":name,"K":p.K,"tau":p.tau,"theta":p.delay,
                        "door_severity":severity,**metrics})
robust=pd.DataFrame.from_records(records)
print(robust.groupby("method")["IAE_c_s"].agg(["median","mean","std"]).round(1))
print("Finished",n_trials,"paired synthetic scenarios.")

## 2. Paired distributions and effect sizes

The **integral absolute error (IAE)** and **total variation** quantify different objectives:

$$\mathrm{IAE}=\sum_{k=0}^{N-1}|r[k]-y[k]|\Delta t,\qquad
\mathrm{TV}(u)=\sum_{k=1}^{N-1}|u[k]-u[k-1]|.\tag{7}$$

*Lower is better*, but overshoot, settling and sensor/actuator constraints matter too. Statistical uncertainty is meaningful only relative to the artificial parameter distributions chosen above.

In [ ]:
piv=robust.pivot(index="trial",columns="method",values="IAE_c_s")
fig,ax=plt.subplots(figsize=(10,3.6))
labels=piv.columns.tolist();ax.boxplot([piv[c].dropna() for c in labels],tick_labels=labels)
ax.set(ylabel="IAE (°C·s)",title="Monte Carlo: synthetic parameter uncertainty")
ax.grid(alpha=.2,axis="y");plt.xticks(rotation=10);plt.tight_layout();plt.show()
for candidate in ["Thesis-sparse-fuzzy","Dense-smoothed-fuzzy"]:
    delta=piv[candidate]-piv["IMC-fixed-nominal"]
    print(candidate,"median paired delta IAE (candidate - IMC):",round(float(delta.median()),2),
          "fraction of scenarios with less IAE:",round(float(np.mean(delta<0)),2))

## 3. Engineering limitations

This single-zone thermal model cannot support certification, actuator safety decisions or claims about spatial uniformity. Real experiments should log four synchronized sensor traces, input power, opening/closing timestamps, ambient temperature, electrical heater constraints and independent validation runs. Use paired seeds only to reduce *synthetic* Monte Carlo sampling noise.

**Next:** [04 · Optimization](04_fuzzy_optimization.ipynb).